# Lab 09: Delta Lake, Pandas API & Spark Connect

## Objectives
- Use Delta Lake features: time travel, OPTIMIZE, Z-ORDER, VACUUM
- Perform MERGE INTO for upsert operations
- Use the Pandas API on Spark for familiar pandas syntax at scale
- Understand Spark Connect architecture and session setup

## Exam Domains
- **Using Spark SQL — 20%**
- **Troubleshooting and Tuning — 10%**
- **Using Pandas API on Apache Spark — 5%**
- **Using Spark Connect — 5%**

## Estimates
- **Time:** ~30 minutes
- **Cost:** ~$1 (serverless)
- **Compute:** Serverless

![Delta Lake Architecture](../assets/diagrams/lab-09-delta-lake-architecture.png)

In [ ]:
USE_CATALOG = "spark_lab_guide"
USE_SCHEMA = "default"

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

## A. Delta Lake — ACID Transactions

Delta Lake stores data as Parquet files with a transaction log (`_delta_log/`). The log tracks every change, enabling ACID transactions, time travel, and audit history.

In [ ]:
# Create a Delta table for experimentation (copy a subset)
spark.sql("""
    CREATE OR REPLACE TABLE taxi_delta AS
    SELECT * FROM taxi_trips LIMIT 10000
""")

# View the table history
spark.sql("DESCRIBE HISTORY taxi_delta").show(truncate=False)

In [ ]:
# Make some changes to create versions
spark.sql("UPDATE taxi_delta SET tip_amount = tip_amount * 1.1 WHERE payment_type = 1")
spark.sql("DELETE FROM taxi_delta WHERE trip_distance < 0.5")

# Now we have 3 versions (0=create, 1=update, 2=delete)
spark.sql("DESCRIBE HISTORY taxi_delta").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

## B. Time Travel

Read previous versions of a Delta table — useful for auditing, debugging, and recovering from mistakes.

In [ ]:
# Read version 0 (original data before any changes)
v0 = spark.sql("SELECT COUNT(*) AS rows FROM taxi_delta VERSION AS OF 0")
v0.show()

# Read current version
current = spark.sql("SELECT COUNT(*) AS rows FROM taxi_delta")
current.show()

# DataFrame API alternative
v0_df = spark.read.option("versionAsOf", 0).table("taxi_delta")
print(f"Version 0 rows: {v0_df.count()}")

In [ ]:
# Restore to a previous version
spark.sql("RESTORE TABLE taxi_delta TO VERSION AS OF 0")
print("Restored to version 0")
spark.sql("SELECT COUNT(*) AS rows FROM taxi_delta").show()
spark.sql("DESCRIBE HISTORY taxi_delta").select("version", "operation").show()

## C. OPTIMIZE and Z-ORDER

Small files slow down reads. `OPTIMIZE` compacts small files into larger ones. `Z-ORDER BY` co-locates related data within files to speed up queries that filter on those columns.

In [ ]:
# Optimize the table — compacts small files
spark.sql("OPTIMIZE taxi_delta")

# Z-Order by PULocationID — co-locates data for faster location-based queries
spark.sql("OPTIMIZE taxi_delta ZORDER BY (PULocationID)")
print("OPTIMIZE and Z-ORDER complete.")

## D. VACUUM

`VACUUM` removes old data files that are no longer referenced by any version. By default, it keeps files for 7 days (168 hours) to support time travel.

In [ ]:
# DRY RUN — shows what files would be deleted (default retention: 168 hours)
spark.sql("VACUUM taxi_delta DRY RUN")

> **Exam Tip:** After running `VACUUM`, you can no longer time-travel to versions older than the retention period. The exam tests whether you understand this trade-off: VACUUM saves storage but limits time travel.

## E. MERGE INTO — Upsert Pattern

`MERGE INTO` combines insert, update, and delete in a single atomic operation. This is the standard pattern for CDC (change data capture) and slowly changing dimensions.

In [ ]:
# MERGE — update matching rows, insert new ones
# First, create a unique updates source using row_number to avoid duplicate matches
spark.sql("""
    CREATE OR REPLACE TEMP VIEW taxi_updates AS
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY tpep_pickup_datetime, PULocationID, DOLocationID
            ORDER BY total_amount
        ) AS rn
        FROM taxi_delta
        LIMIT 200
    )
    WHERE rn = 1
    LIMIT 100
""")

spark.sql("""
    MERGE INTO taxi_delta AS target
    USING (
        SELECT tpep_pickup_datetime, PULocationID, DOLocationID,
               tip_amount + 5.0 AS new_tip, total_amount + 5.0 AS new_total
        FROM taxi_updates
    ) AS source
    ON target.tpep_pickup_datetime = source.tpep_pickup_datetime
       AND target.PULocationID = source.PULocationID
       AND target.DOLocationID = source.DOLocationID
    WHEN MATCHED THEN UPDATE SET
        target.tip_amount = source.new_tip,
        target.total_amount = source.new_total
""")

print("MERGE complete.")
spark.sql("DESCRIBE HISTORY taxi_delta").select("version", "operation", "operationMetrics").show(5, truncate=False)

## F. Pandas API on Spark

The Pandas API on Spark (`pyspark.pandas`) lets you write familiar pandas code that runs on Spark — scaling to large datasets without rewriting your code. It is NOT the same as `toPandas()` (which collects all data to the driver).

In [ ]:
import pyspark.pandas as ps

# Read a Delta table as a pandas-on-Spark DataFrame
taxi_ps = ps.read_table("taxi_trips")

# Familiar pandas syntax — runs on Spark cluster, not driver
print(f"Shape: {taxi_ps.shape}")
print(f"Columns: {list(taxi_ps.columns)}")
taxi_ps.head()

In [ ]:
# pandas-style operations — distributed under the hood
taxi_ps["trip_distance"].describe()

In [ ]:
# GroupBy — same as pandas, runs on Spark
taxi_ps.groupby("payment_type")["total_amount"].mean()

In [ ]:
# Convert between Spark DataFrame and pandas-on-Spark DataFrame
spark_df = taxi_ps.to_spark()  # pandas-on-Spark -> Spark DataFrame
ps_df = spark_df.pandas_api()  # Spark DataFrame -> pandas-on-Spark

print(f"Spark DataFrame type: {type(spark_df)}")
print(f"Pandas-on-Spark type: {type(ps_df)}")

> **Exam Tip:** `pyspark.pandas` (Pandas API on Spark) runs distributed on the cluster. `toPandas()` collects ALL data to the driver. The exam tests this distinction — using `toPandas()` on large data causes OutOfMemoryError, while `pyspark.pandas` scales.

## G. Spark Connect

Spark Connect is a thin client protocol that decouples the client from the Spark cluster. Instead of running the driver inside the cluster, the client sends plans over gRPC and receives results back. This enables remote connectivity from any application.

In [ ]:
# In Databricks, you're already connected via Spark Connect by default on serverless
# The SparkSession is configured to use the remote connection

# Check the connection type
print(f"Spark version: {spark.version}")
print(f"Session type: {type(spark)}")

# Spark Connect session creation (conceptual — for connecting from outside Databricks):
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.remote("sc://hostname:443/;token=<token>").getOrCreate()

In [ ]:
# Spark Connect uses the same DataFrame API — no code changes needed
# This query runs identically whether using Spark Connect or classic Spark
result = (
    spark.table("taxi_trips")
    .groupBy("PULocationID")
    .count()
    .orderBy("count", ascending=False)
    .limit(5)
)
result.show()

> **Exam Tip:** Spark Connect does NOT change the DataFrame API. Your existing PySpark code works without modification. The key difference is the connection method: `SparkSession.builder.remote("sc://host:port")` instead of `.master("local")` or cluster deployment.

## Exam-Style Review

**Q1.** How do you read version 3 of a Delta table named `my_table`?
- A) `spark.read.option("version", 3).table("my_table")`
- B) `spark.read.option("versionAsOf", 3).table("my_table")`
- C) `spark.sql("SELECT * FROM my_table@v3")`
- D) `spark.read.version(3).table("my_table")`

**Q2.** What happens after running `VACUUM my_table RETAIN 0 HOURS`?
- A) Nothing — VACUUM has no effect
- B) All old data files are deleted and time travel to previous versions is no longer possible
- C) Only the transaction log is deleted
- D) The table is dropped

**Q3.** What is the difference between `pyspark.pandas` and `toPandas()`?
- A) They are identical
- B) `pyspark.pandas` runs distributed on the cluster; `toPandas()` collects all data to the driver
- C) `toPandas()` is faster because it uses the driver's memory
- D) `pyspark.pandas` requires a different API than pandas

**Q4.** What protocol does Spark Connect use for client-server communication?
- A) REST API
- B) JDBC
- C) gRPC
- D) WebSocket

**Q5.** What does `OPTIMIZE table ZORDER BY (col)` do?
- A) Sorts the table by the column
- B) Creates an index on the column
- C) Compacts small files and co-locates data by the column for faster filtering
- D) Partitions the table by the column

### Answers
- **Q1: B** — Use `option("versionAsOf", 3)` or SQL `VERSION AS OF 3`.
- **Q2: B** — VACUUM with 0-hour retention deletes ALL old files, permanently removing time travel capability for past versions.
- **Q3: B** — `pyspark.pandas` runs distributed. `toPandas()` brings everything to the driver (OOM risk on large data).
- **Q4: C** — Spark Connect uses gRPC for client-server communication.
- **Q5: C** — Z-ORDER co-locates related data within files (data skipping optimization), and OPTIMIZE compacts small files.

## Key Takeaways
- Delta Lake provides ACID transactions, time travel, OPTIMIZE, Z-ORDER, and VACUUM
- `DESCRIBE HISTORY` shows the audit log of all changes
- VACUUM removes old files but limits time travel to the retention period
- `MERGE INTO` is the standard upsert pattern (insert + update + delete in one operation)
- `pyspark.pandas` runs distributed (safe for large data); `toPandas()` collects to driver (OOM risk)
- Spark Connect uses gRPC to decouple the client from the cluster — same API, remote execution

**Next:** [Lab 10 — Performance Tuning & Spark UI](10-performance-tuning-spark-ui.ipynb)

In [ ]:
spark.sql("DROP TABLE IF EXISTS taxi_delta")
spark.catalog.dropTempView("taxi_updates")
print("Lab 09 cleanup complete.")